# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [1]:
def lettre_vers_nombre(c):
    """Numéro de 0 à 25 d'une lettre majuscule."""
    return ord(c) - ord("A")

def nombre_vers_lettre(n):
    """Lettre majuscule correspondant à un numéro de 0 à 25."""
    return chr(n % 26 + ord("A"))

print(lettre_vers_nombre("C"), nombre_vers_lettre(2))
print([nombre_vers_lettre(lettre_vers_nombre(c)) == c for c in "ABCXYZ"])


2 C
[True, True, True, True, True, True]


## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [2]:
import unicodedata

def nettoyer(texte):
    """Majuscules sans accents, lettres A-Z uniquement."""
    sans_accent = unicodedata.normalize("NFD", texte)
    sans_accent = "".join(c for c in sans_accent if unicodedata.category(c) != "Mn")
    return "".join(c for c in sans_accent.upper() if "A" <= c <= "Z")

print(nettoyer("Été, à Paris !"))

ETEAPARIS


## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [3]:
def chiffrer(texte, a, b):
    """Chiffrement affine de texte avec la clé (a, b)."""
    resultat = []
    for c in nettoyer(texte):
        x = lettre_vers_nombre(c)
        resultat.append(nombre_vers_lettre((a * x + b) % 26))
    return "".join(resultat)

message = "ATTAQUEALAUBE"
print(chiffrer(message, 5, 8))
print(chiffrer(message, 1, 3))


IZZIKECILIENC
DWWDTXHDODXEH


## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [4]:
import math

valides = [a for a in range(1, 26) if math.gcd(a, 26) == 1]
print(valides, len(valides))


[1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25] 12


## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [5]:
def inverse_modulaire(a, n):
    for x in range(1, n):
        if a * x % n == 1:
            return x
    raise ValueError("pas d'inverse")

print(inverse_modulaire(5, 26))


21


## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [6]:
def dechiffrer(texte, a, b):
    inverse = inverse_modulaire(a, 26)
    return "".join(nombre_vers_lettre(inverse * (lettre_vers_nombre(c) - b) % 26)
                   for c in nettoyer(texte))

m = "ATTAQUE A L'AUBE"
print(dechiffrer(chiffrer(m, 5, 8), 5, 8) == nettoyer(m))


True


## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [7]:
secret = chiffrer("Rendez-vous demain devant la gare, apportez le message secret", 7, 3)

def toutes_les_clefs(secret):
    return [((a, b), dechiffrer(secret, a, b)) for a in valides for b in range(26)]

propositions = toutes_les_clefs(secret)
print(len(propositions))
for cle, clair in propositions:
    print(cle, clair)

312
(1, 0) SFQYFWUXNZYFJDHQYFUDQGCDTDSFDEEXSGFWCFJFZZDTFZFRSFG
(1, 1) REPXEVTWMYXEICGPXETCPFBCSCRECDDWRFEVBEIEYYCSEYEQREF
(1, 2) QDOWDUSVLXWDHBFOWDSBOEABRBQDBCCVQEDUADHDXXBRDXDPQDE
(1, 3) PCNVCTRUKWVCGAENVCRANDZAQAPCABBUPDCTZCGCWWAQCWCOPCD
(1, 4) OBMUBSQTJVUBFZDMUBQZMCYZPZOBZAATOCBSYBFBVVZPBVBNOBC
(1, 5) NALTARPSIUTAEYCLTAPYLBXYOYNAYZZSNBARXAEAUUYOAUAMNAB
(1, 6) MZKSZQORHTSZDXBKSZOXKAWXNXMZXYYRMAZQWZDZTTXNZTZLMZA
(1, 7) LYJRYPNQGSRYCWAJRYNWJZVWMWLYWXXQLZYPVYCYSSWMYSYKLYZ
(1, 8) KXIQXOMPFRQXBVZIQXMVIYUVLVKXVWWPKYXOUXBXRRVLXRXJKXY
(1, 9) JWHPWNLOEQPWAUYHPWLUHXTUKUJWUVVOJXWNTWAWQQUKWQWIJWX
(1, 10) IVGOVMKNDPOVZTXGOVKTGWSTJTIVTUUNIWVMSVZVPPTJVPVHIVW
(1, 11) HUFNULJMCONUYSWFNUJSFVRSISHUSTTMHVULRUYUOOSIUOUGHUV
(1, 12) GTEMTKILBNMTXRVEMTIREUQRHRGTRSSLGUTKQTXTNNRHTNTFGTU
(1, 13) FSDLSJHKAMLSWQUDLSHQDTPQGQFSQRRKFTSJPSWSMMQGSMSEFST
(1, 14) ERCKRIGJZLKRVPTCKRGPCSOPFPERPQQJESRIORVRLLPFRLRDERS
(1, 15) DQBJQHFIYKJQUOSBJQFOBRNOEODQOPPIDRQHNQUQKKOEQKQCDQR
(1, 16) CPAIPGEHXJIPTNRAIPENAQMNDNCPNOOHCQPGMP

## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [8]:
from collections import Counter

y = lettre_vers_nombre(Counter(secret).most_common(1)[0][0])
candidats = [(a, (y - 4 * a) % 26) for a in valides]

def score(clair):
    return sum(clair.count(c) for c in "ESAINT")

meilleur = max(candidats, key=lambda cle: score(dechiffrer(secret, *cle)))
print(len(candidats), meilleur, dechiffrer(secret, *meilleur))


12 (7, 3) RENDEZVOUSDEMAINDEVANTLAGAREAPPORTEZLEMESSAGESECRET


## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [9]:
import random

def indice_coincidence(texte):
    N = len(texte)
    return sum(n * (n - 1) for n in Counter(texte).values()) / (N * (N - 1))

texte_long = nettoyer("Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.")
secret_long = chiffrer(texte_long, 5, 8)
hasard = "".join(random.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in texte_long)

print(indice_coincidence(texte_long), indice_coincidence(secret_long), indice_coincidence(hasard))


0.06887003652645222 0.06887003652645222 0.039781430422999885


## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [10]:
def vigenere(texte, cle):
    texte = nettoyer(texte)
    return "".join(nombre_vers_lettre(lettre_vers_nombre(c) + lettre_vers_nombre(cle[i % len(cle)]))
                   for i, c in enumerate(texte))

chiffre_vig = vigenere(texte_long, "CLE")
print(indice_coincidence(chiffre_vig))


0.0474402026628962


## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [11]:
for L in range(1, 8):
    colonnes = [chiffre_vig[i::L] for i in range(L)]
    moyenne = sum(indice_coincidence(c) for c in colonnes) / L
    print(L, round(moyenne, 3))

1 0.047
2 0.047
3 0.068
4 0.05
5 0.049
6 0.068
7 0.045
